In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

This notebook implements an array-based version of Heapsort.

# Heapsort

The function call `swap(A, i, j)` takes an array `A` and  two indexes `i` and `j` and exchanges the elements at these indexes.

In [ ]:
function swap(A: number[], i: number, j: number): void {
  const t = A[i];
  A[i] = A[j];
  A[j] = t;
}

The procedure `sink` takes three arguments.
- `A` is the array representing the heap.
- `k` is an index into the array `A`.
- `n` is the upper bound  of the part of this array that has to be transformed into a heap.  

The array `A` itself might actually have more than $n+1$ elements, but for the
purpose of the method `sink` we restrict our attention to the subarray
`A[k:n]`. 
When calling `sink`, the assumption is that `A[k:n+1]` should represent a heap 
that possibly has its heap condition violated at its root, i.e. at index `k`.  The
purpose of the procedure `sink` is to restore the heap condition at index `k`.
- We compute the index `j` of the left subtree below index `k`.
- We check whether there also is a right subtree at position `j+1`.
  
  This is the case if `j + 1 <= n`.  
- If the heap condition is violated at index `k`, we exchange the element at  position `k` 
  with the child that has the higher priority, i.e. the child that is smaller. 
- Next, we check in line 9 whether the heap condition is violated at index `k`.  
  If the heap condition is satisfied, there is nothing left to do and the procedure returns.  
  
- Otherwise, the element at position `k` is swapped with
  the element at position `j`.  
  
  Of course, after this swap it is possible that the heap condition is
  violated at position `j`.  Therefore,  `k` is set to `j` and the `while`-loop continues
  as long as the node at position `k` has at least one child, i.e. as long as 
  `2 * k + 1 <= n`.

In [ ]:
function sink(A: number[], k: number, n: number): void {
  while (2 * k + 1 <= n) {
    let j = 2 * k + 1;
    if (j + 1 <= n && A[j] > A[j + 1]) j += 1;
    if (A[k] <= A[j]) return;
    swap(A, k, j);
    k = j;
  }
}

The function call `heapSort(A)` has the task to sort the array `A` and proceeds in two phases.
- In phase one our goal is to transform the array `A`into a heap that is stored in `A`.

  In order to do so, we traverse the array `A` in reverse in a loop.  
  The invariant of this loop is that before 
  `sink` is called, all trees rooted at an index greater than 
  `k` satisfy the heap condition.  Initially this is true because the trees that
  are rooted at indices greater than  $(n + 1) // 2 - 1$ are trivial, i.e. they only
  consist of their root node.  
            
  In order to maintain the invariant for index `k`, `sink` is called with
  argument `k`,  since at this point, the tree rooted at index `k` satisfies
  the heap condition except possibly at the root.  It is then the job of $\texttt{sink}$ to
  establish the heap condition at index `k`.  If the element at the root has a
  priority that is too low, `sink` ensures that this element sinks down in the tree
  as far as necessary.
- In phase two we remove the elements from the heap one-by-one and insert them at the end of
  the array.

  When the `while`-loop starts, the array `A` contains a heap.  Therefore,
  the smallest element is found at the root of the heap.  Since we want to sort the
  array `A` descendingly, we move this element to the end of the array `A` and in
  return move the element from the end of the array`A`to the front.
  After this exchange, the subarray `A[0:n-1]` represents a heap, except that the
  heap condition might now be violated at the root.  Next, we decrement `n`, since the
  last element of the array `A` is already in its correct position.  
  In order to reestablish the heap condition at the root, we call `sink` with index
  `0`.

In [ ]:
function heapSort(A: number[]): void {
  let n = A.length - 1;

  for (let k = Math.floor((n + 1) / 2) - 1; k >= 0; k--) {
    sink(A, k, n);
  }

  while (n >= 1) {
    swap(A, 0, n);
    n -= 1;
    sink(A, 0, n);
  }
}

## Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

The function `isOrdered(L)` checks that the array `L` is sorted descendingly.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] < L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `true` if the array `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => Math.floor(Math.random() * (2 * k)));
    const oldL = L.slice();
    heapSort(L);
    isOrdered(L);
    sameElements(L, oldL);
    process.stdout.write(".");
  }
  console.log();
  console.log("All tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 20_000);
console.timeEnd("testSort");

Next, we sort a million random integers.  It is not as fast as merge sort, but we do not need an auxiliary array and hence we don't need additional storage.

In [ ]:
console.time("heapSort 1,000,000");
const k = 1_000_000;
const L = Array.from({ length: k }, () => Math.floor(Math.random() * (2 * k)));
heapSort(L);
console.timeEnd("heapSort 1,000,000");